In [4]:
import torch
import numpy as np
import matplotlib.pyplot as plt

# read data
import sys
sys.path.append('../..')
import data_preprocessing
# import data_preprocessing.coil20 as coil20
# import data_preprocessing.cifar10 as cifar10
# import data_preprocessing.reuters as reuters
# import data_preprocessing.tdt2 as tdt2
from importlib import reload

# training process
import algorithm.algo1 as algo1
reload(algo1)

# evaluation
import algorithm.kmeans as kmeans
import eval_metrics.ac
import eval_metrics.nmi

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('[Using device]', device)

[Using device] cuda


In [ ]:
import argparse

parser = argparse.ArgumentParser(description='Train a model for clustering')
parser.add_argument('--n_clusters', type=int, default=20, help='number of clusters')
parser.add_argument('--loss', type=str, default='reconstruction_loss_without_regularization', help='loss function')
parser.add_argument('--lr', type=float, default=1e-4, help='learning rate')
parser.add_argument('--lamda', type=float, default=1, help='regularization parameter')
args = parser.parse_args()

CLUSTER_NUM = args.n_clusters
LOSS_FUNCTION_NAME = args.loss
LOSS_FUNCTION = getattr(origin, LOSS_FUNCTION_NAME)
LEARNING_RATE = args.lr
if LOSS_FUNCTION_NAME == 'reconstruction_loss_without_regularization':
    LAMBDA = None
else:
    LAMBDA = args.lamda

In [9]:
CLUSTER_NUM = 10
LAMBDA = 5
import importlib

ds = 'tdt2'
module_name = f'data_preprocessing.{ds}'
DATASET = importlib.import_module(module_name)

# k = 10, lambda=10 (越大越不易nan); k-means多几次可达79%

In [3]:
# read data
mat, label = DATASET.read_data(num_of_class=CLUSTER_NUM, random_shuffle=True, 
                              random_choose_class=True)
mat = torch.tensor(mat, dtype=torch.float32)
mat -= torch.mean(mat)

print(f'[Using classes] {np.unique(np.array(label))}')

# compute the similarity matrix using cosine similarity
mat_normalized = mat / torch.norm(mat, dim=1).reshape(-1, 1)
A = mat_normalized @ mat_normalized.T
A.to(device)
pass

[Using classes] [0 1 2 3 4 5 6 7 8 9]


In [ ]:
reload(algo1)
H = torch.randn(A.shape[0], CLUSTER_NUM) + 1
H.to(device)

H_out, P_out = algo1.algo1(A=A, H=H, 
                           lamda=LAMBDA, 
                           max_iter=10000000,
                           tol=1e-15, print_every=100)

In [ ]:
reload(kmeans)
pred_labels, centroids = kmeans.kmeans_optimal(H_out, CLUSTER_NUM, turns=40)
acc = eval_metrics.ac.ac(label, pred_labels)
nmi = eval_metrics.nmi.nmi(label, pred_labels)
print(f'[Accuracy] {acc}\n[NMI] {nmi}')